# Daily Practice — 2026-09-22 — ML Testing & MLOps: Building a Slice-Based Model Evaluation Test Suite

**Dataset:** [Diamonds](https://ggplot2.tidyverse.org/reference/diamonds.html) — 53,940 real diamond
sales records (carat, cut, color, clarity, dimensions, price), loaded from the `mwaskom/seaborn-data`
mirror of the classic `ggplot2` diamonds dataset.

## Problem statement

A regression model has been trained to predict a diamond's `price` from its physical and quality
attributes (`carat`, `cut`, `color`, `clarity`, `depth`, `table`, `x`, `y`, `z`). The aggregate test-set
MAE looks healthy, and a release engineer signs off on "MAE below $300" as the ship gate.

As the QA/MLOps engineer, you don't trust a single aggregate number to tell the whole story — a model
can look fine on average while quietly failing on specific, commercially important segments (e.g. the
rarest, lowest-quality stones, which are exactly the ones a jeweler is most likely to escalate a bad
quote on). This is the regression-model analogue of a classic QA problem: a test suite that only checks
overall pass rate will miss a feature that's completely broken for one customer segment.

Your job is to build a **slice-based evaluation test suite**: a reusable set of checks that break the
model's error down by category (`cut`, `color`, `clarity`) and fail the build if any *sufficiently large*
slice's error is much worse than the overall average — even when the aggregate metric alone would pass.

## What you should produce

1. A train/test split that keeps the categorical quality columns (`cut`, `color`, `clarity`) available
   for slicing the *test* set afterward, a fitted `RandomForestRegressor` baseline, and its overall MAE
   and MAPE (mean absolute percentage error) on the held-out test set.
2. A `slice_performance_report(y_true, y_pred, slice_series, min_slice_size=50)` function that groups
   the held-out predictions by a categorical column, computes `n`, `mae`, and `mape` per slice, drops
   any slice with fewer than `min_slice_size` samples (too small to trust), and returns the result
   sorted worst-to-best by `mape`.
3. A `slice_gate(report, overall_mape, max_relative_degradation=1.5)` function that takes a slice report
   and flags every slice whose `mape` exceeds `overall_mape * max_relative_degradation`, returning a
   dict with `passed`, `threshold`, and the `failing_slices` DataFrame — the kind of function you'd wire
   into a CI/release pipeline as an automated gate.
4. Slice reports and gate results for all three categorical columns (`cut`, `color`, `clarity`).
5. A short written verdict (3–5 sentences): which slice(s) actually fail the 1.5x gate, which come close
   without failing, why the aggregate MAE/MAPE alone would have shipped this model without objection,
   and what `min_slice_size` is doing for you (what would happen without it).

Try it yourself in the starter cells below before you scroll down to the solution.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42
pd.set_option("display.width", 120)

## Load the data

In [ ]:
DIAMONDS_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"

df = pd.read_csv(DIAMONDS_URL)

# Quality columns are ordinal categories, not free-text categoricals -- encode them as ordered
# integer codes (worst -> best) so the model can use the ordering, but keep the *original* string
# columns around separately for slicing the test set later.
CUT_ORDER = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
COLOR_ORDER = ["J", "I", "H", "G", "F", "E", "D"]
CLARITY_ORDER = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]

df_enc = df.copy()
df_enc["cut"] = pd.Categorical(df_enc["cut"], categories=CUT_ORDER, ordered=True).codes
df_enc["color"] = pd.Categorical(df_enc["color"], categories=COLOR_ORDER, ordered=True).codes
df_enc["clarity"] = pd.Categorical(df_enc["clarity"], categories=CLARITY_ORDER, ordered=True).codes

FEATURES = ["carat", "cut", "color", "clarity", "depth", "table", "x", "y", "z"]
TARGET = "price"
SLICE_COLS = ["cut", "color", "clarity"]  # original string columns, used only for slicing

print(df.shape)
df[FEATURES + [TARGET]].describe()

## Your task

Fill in the TODOs below. Function signatures are given; the logic is up to you.

In [ ]:
# TODO: split X, y, and the original (string) slice columns into train/test sets together, so that
# slices_test lines up row-for-row with y_test and your model's predictions on X_test.
#
# X = df_enc[FEATURES]
# y = df_enc[TARGET]
# slices = df[SLICE_COLS]
#
# X_train, X_test, y_train, y_test, slices_train, slices_test = train_test_split(
#     ..., test_size=0.2, random_state=RANDOM_STATE
# )


In [ ]:
# TODO: fit a RandomForestRegressor baseline on (X_train, y_train) and compute its overall
# MAE and MAPE on (X_test, y_test). Store predictions as `pred_test` -- you'll reuse them below.
#
# model = RandomForestRegressor(...)
# model.fit(...)
# pred_test = model.predict(X_test)
# overall_mae = ...
# overall_mape = ...


In [ ]:
def slice_performance_report(y_true, y_pred, slice_series, min_slice_size=50):
    """Group (y_true, y_pred) by slice_series and return per-slice n / mae / mape,
    dropping slices smaller than min_slice_size, sorted worst-to-best by mape.

    Returns a DataFrame with columns: slice, n, mae, mape.
    """
    # TODO:
    # 1. Compute per-row absolute error and absolute percentage error.
    # 2. Group by slice_series, aggregating n (count), mae (mean abs error), mape (mean ape).
    # 3. Drop groups with n < min_slice_size.
    # 4. Sort by mape descending and return with a reset index.
    raise NotImplementedError

In [ ]:
def slice_gate(report, overall_mape, max_relative_degradation=1.5):
    """Flag every slice in `report` whose mape exceeds
    overall_mape * max_relative_degradation.

    Returns a dict: {"passed": bool, "threshold": float, "failing_slices": DataFrame}.
    """
    # TODO:
    # 1. Compute threshold = overall_mape * max_relative_degradation.
    # 2. Select rows of `report` whose mape exceeds threshold.
    # 3. passed = True iff no rows exceed it.
    raise NotImplementedError

In [ ]:
# TODO: for each column in SLICE_COLS, call slice_performance_report and slice_gate, and
# print both the report and whether the gate passed.
#
# for col in SLICE_COLS:
#     report = slice_performance_report(y_test, pred_test, slices_test[col])
#     gate = slice_gate(report, overall_mape)
#     ...


### Write-up

TODO: in 3–5 sentences, say which slice(s) actually fail the 1.5x gate, which come close without
failing, why the aggregate MAE/MAPE alone would have shipped this model without objection, and what
`min_slice_size` is doing for you (what would happen without it).

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
# 1. Train/test split, keeping slice columns aligned with the test set
X = df_enc[FEATURES]
y = df_enc[TARGET]
slices = df[SLICE_COLS]

X_train, X_test, y_train, y_test, slices_train, slices_test = train_test_split(
    X, y, slices, test_size=0.2, random_state=RANDOM_STATE
)

# 2. Baseline model + overall metrics
model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1)
model.fit(X_train, y_train)
pred_test = model.predict(X_test)

overall_mae = mean_absolute_error(y_test, pred_test)
overall_mape = float(np.mean(np.abs(pred_test - y_test) / y_test))
print(f"overall_mae={overall_mae:.2f}  overall_mape={overall_mape:.4f}")

# 3. Slice performance report
def slice_performance_report(y_true, y_pred, slice_series, min_slice_size=50):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    slice_series = pd.Series(np.asarray(slice_series), name="slice")
    abs_err = np.abs(y_true - y_pred)
    ape = abs_err / y_true
    frame = pd.DataFrame({"slice": slice_series.values, "abs_err": abs_err, "ape": ape})
    report = frame.groupby("slice").agg(n=("abs_err", "size"), mae=("abs_err", "mean"), mape=("ape", "mean"))
    report = report[report["n"] >= min_slice_size].sort_values("mape", ascending=False)
    return report.reset_index()

# 4. Slice gate
def slice_gate(report, overall_mape, max_relative_degradation=1.5):
    threshold = overall_mape * max_relative_degradation
    failing = report[report["mape"] > threshold].copy()
    failing["threshold"] = threshold
    return {
        "passed": failing.empty,
        "threshold": threshold,
        "failing_slices": failing.reset_index(drop=True),
    }

# 5. Run the suite across all three categorical columns
for col in SLICE_COLS:
    print(f"=== {col} ===")
    report = slice_performance_report(y_test, pred_test, slices_test[col], min_slice_size=50)
    print(report)
    gate = slice_gate(report, overall_mape, max_relative_degradation=1.5)
    print("gate passed:", gate["passed"])
    if not gate["passed"]:
        print(gate["failing_slices"])
    print()
```

**Typical output**

```
overall_mae=275.89  overall_mape=0.0737

=== cut ===
       slice     n         mae      mape
0       Fair   335  481.999867  0.110411
1       Good  1004  283.810789  0.078075
2      Ideal  4292  236.420770  0.074293
3    Premium  2775  338.168698  0.074055
4  Very Good  2382  242.136028  0.065238
gate passed: True

=== color ===
  slice     n         mae      mape
0     D  1362  249.151686  0.078665
1     I  1148  362.061895  0.078104
2     H  1597  340.333947  0.077995
3     E  1934  216.697734  0.072097
4     G  2277  264.138399  0.070657
5     F  1898  243.612277  0.070344
6     J   572  340.738587  0.069834
gate passed: True

=== clarity ===
  slice     n         mae      mape
0    I1   156  452.426543  0.118361
1   SI2  1888  431.323552  0.081534
2  VVS1   711  163.024454  0.074219
3   VS2  2465  252.687874  0.073196
4   SI1  2545  280.541180  0.072823
5  VVS2  1022  193.558009  0.068668
6   VS1  1641  229.075836  0.067879
7    IF   360  180.304587  0.062892
gate passed: False
  slice    n         mae      mape  threshold
0    I1  156  452.426543  0.118361   0.110559
```

**Explanation**

Only one slice actually trips the 1.5x gate: `clarity == I1` (the worst clarity grade, only 156 test
rows), with MAPE 0.118 against a threshold of 0.111 — a real, if narrow, fail. `cut == Fair` comes very
close (MAPE 0.110, just under its own threshold of ~0.111) without failing, which is exactly the kind
of near-miss a good test suite surfaces for a human to look at rather than silently ignoring. Every
`color` slice passes comfortably. None of this is visible from the aggregate numbers: overall MAE
($276) and MAPE (7.4%) look uniformly healthy, and a release gate built only around them would ship
this model without a second look — the failure only exists at the intersection of "worst quality grade"
and "smallest sample size," which is precisely where a model has the least training signal and the
segment's stakeholders (jewelers pricing low-clarity stones) are the ones most likely to notice.
`min_slice_size` is what keeps this suite honest: without it, tiny categories with a handful of test
rows would produce wildly noisy MAPE estimates (a single bad prediction on an 8-row slice can double its
MAPE) and the gate would be swamped with spurious failures — exactly the flaky-test problem from
traditional QA, just moved into model evaluation.

</details>